# VietSight — chuẩn bị dữ liệu trên Kaggle (vietsight_v1)
Kaggle: **Accelerator: None (CPU)**, **Internet: On**. Run all. Thời gian: tải ~21 GB stream Mapillary + chuyển đổi (~30–60 phút).
Kết quả (Output của notebook): `versions/vietsight_v1/` (images/labels train+val, `data.yaml`) + `versions/vietsight_v1_report/` + `val_frozen.txt`.
Sau đó ở notebook huấn luyện: *Add Input → Notebook Output → notebook này*.
Code repo được nhúng (sha256 `b5053a87fa00b6d1…`, 27 tệp) — sinh bởi `scripts/make_kaggle_notebooks.py`, không sửa tay.

In [ ]:
VERSION_CONFIG = "configs/versions/v1_first_gear.yaml"
MAPILLARY_OTHER_FRAC = 0.3   # tỉ lệ ảnh KHÔNG có xe 2 bánh/người lái được giữ (ảnh có thì luôn giữ)
WORKERS = 4
RAW = "/tmp/raw"            # dữ liệu gốc: ổ tạm, không vào Output
PROCESSED = "/tmp/processed"
OUT_ROOT = "/kaggle/working/versions"
FETCH = True                # False = dùng dữ liệu đã có trong RAW (chạy lại nhanh)
MAX_ANN = 0                 # >0: chỉ đọc N annotation đầu (thử nhanh)

In [ ]:
import base64, hashlib, io, os, sys, zipfile, subprocess
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "b5053a87fa00b6d144af695034d25409cfd54b4ca2d2f46abd75063aa338fd0f", "bundle hỏng"
REPO = os.path.abspath("VietSight")  # Kaggle: /kaggle/working/VietSight
zipfile.ZipFile(io.BytesIO(raw)).extractall(REPO)
os.chdir(REPO)
print("code bundle b5053a87fa00b6d1 ->", REPO)

In [ ]:
!"{sys.executable}" -m pip install -q opencv-python-headless pycocotools pyyaml tqdm requests 2>&1 | tail -2

In [ ]:
if FETCH:
    !"{sys.executable}" scripts/fetch_sly_tar.py mapillary_vistas {RAW}/mapillary_vistas --splits training validation --prefer motorcycle motorcyclist bicycle bicyclist "other rider" "other vehicle" "wheeled slow" --other-frac {MAPILLARY_OTHER_FRAC} --max-ann {MAX_ANN} --max-side 1024
!df -h /tmp /kaggle/working || true

In [ ]:
!"{sys.executable}" scripts/convert_supervisely.py --config mapillary_vistas --root {RAW}/mapillary_vistas --workers {WORKERS} --out {PROCESSED}

In [ ]:
# Nguồn đã xử lý sẵn được attach (vd. dữ liệu VN của nhóm, đóng gói bằng scripts/package_kaggle.py source <tên>)
import glob, shutil, os
for meta in glob.glob("/kaggle/input/**/processed/*/meta.jsonl", recursive=True):
    src = os.path.dirname(meta)
    dst = os.path.join(PROCESSED, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print("nguồn đã xử lý:", src, "->", dst)

In [ ]:
!"{sys.executable}" scripts/build_version.py {VERSION_CONFIG} --processed {PROCESSED} --out-root {OUT_ROOT}

In [ ]:
import shutil
try:
    from IPython.display import Image, display
except ImportError:  # plain python (local test)
    Image, display = (lambda filename: filename), print
shutil.copyfile("configs/val_frozen.txt", f"{OUT_ROOT}/val_frozen.txt")
for p in ("preview_train.png", "preview_val.png"):
    display(Image(filename=f"{OUT_ROOT}/vietsight_v1_report/" + p))
print(open(f"{OUT_ROOT}/vietsight_v1_report/notebook_report.txt", encoding="utf-8").read())
print("TẢI VỀ và commit: versions/val_frozen.txt -> configs/val_frozen.txt, versions/vietsight_v1_report/stats.json -> data/versions/")